# Expandiffusion on Google Colab

This notebook installs Expandiffusion, starts the FastAPI backend and Vite frontend, verifies `/api/health`, and prints the temporary Colab URL for the web UI. The URL only works while this Colab runtime is active.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/PailletJuanPablo/expandiffusion.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/expandiffusion")
API_PORT = 8011
WEB_PORT = 5180

In [ ]:
import subprocess
import sys

print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=False)

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA devices:", torch.cuda.device_count())
except Exception as exc:
    print("Torch is not ready yet:", exc)

In [ ]:
import subprocess

if not REPO_URL:
    raise ValueError("Set REPO_URL to the hosted Git repository before running this cell.")

if (PROJECT_DIR / "apps" / "api").exists():
    print(f"Using existing checkout at {PROJECT_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

In [ ]:
import subprocess
import sys

api_dir = PROJECT_DIR / "apps" / "api"
web_dir = PROJECT_DIR / "apps" / "web"

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev,diffusers]"], cwd=api_dir, check=True)
subprocess.run(["npm", "--prefix", str(web_dir), "ci"], check=True)

In [ ]:
import os
import sys

def read_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

hf_token = os.environ.get("HF_TOKEN", "") or read_colab_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

env_values = {
    "EXPANDIFFUSION_PLUGIN_DIR": "apps/api/plugins",
    "EXPANDIFFUSION_API_HOST": "0.0.0.0",
    "EXPANDIFFUSION_API_PORT": str(API_PORT),
    "EXPANDIFFUSION_WEB_HOST": "0.0.0.0",
    "EXPANDIFFUSION_WEB_PORT": str(WEB_PORT),
    "EXPANDIFFUSION_WEB_ALLOWED_HOSTS": "*",
    "EXPANDIFFUSION_PYTHON": sys.executable,
}
if hf_token:
    env_values["HF_TOKEN"] = hf_token

for key, value in env_values.items():
    os.environ[key] = value

env_text = "\n".join(f"{key}={value}" for key, value in env_values.items()) + "\n"
(PROJECT_DIR / ".env").write_text(env_text, encoding="utf-8")
print("Colab environment written. HF_TOKEN set:", bool(hf_token))

In [ ]:
import json
import os
import subprocess
import time
import urllib.request

def wait_for_json(url, label, timeout_seconds=180):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                body = response.read().decode("utf-8")
                return json.loads(body)
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"{label} did not become ready at {url}: {last_error}")

def wait_for_text(url, label, timeout_seconds=180):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                return response.status, response.read(256).decode("utf-8", errors="replace")
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"{label} did not become ready at {url}: {last_error}")

if "EXPANDIFFUSION_PROCESS" in globals() and EXPANDIFFUSION_PROCESS.poll() is None:
    EXPANDIFFUSION_PROCESS.terminate()
    EXPANDIFFUSION_PROCESS.wait(timeout=20)

log_dir = PROJECT_DIR / "colab-logs"
log_dir.mkdir(exist_ok=True)
log_file = open(log_dir / "dev.log", "w", encoding="utf-8")
EXPANDIFFUSION_PROCESS = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd=PROJECT_DIR,
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)

api_health = wait_for_json(f"http://127.0.0.1:{API_PORT}/api/health", "Expandiffusion API")
web_status, web_preview = wait_for_text(f"http://127.0.0.1:{WEB_PORT}/", "Expandiffusion web UI")

print("Dev process PID:", EXPANDIFFUSION_PROCESS.pid)
print("API health:", api_health)
print("Local web status:", web_status)
print("Local web preview:", web_preview[:120])

In [ ]:
import json
from IPython.display import HTML, display
from google.colab import output

health_url = f"http://127.0.0.1:{API_PORT}/api/health"
api_health = wait_for_json(health_url, "Expandiffusion API")
if not api_health.get("ok"):
    raise RuntimeError(f"API health check failed: {api_health}")

ui_url = output.eval_js(f"google.colab.kernel.proxyPort({WEB_PORT})")
if not isinstance(ui_url, str) or not ui_url.startswith("http"):
    raise RuntimeError(f"Colab did not return a valid proxy URL: {ui_url!r}")

proxy_result = output.eval_js(
    """
    (async () => {
      try {
        const response = await fetch(%s, {cache: 'no-store'});
        return {ok: response.ok, status: response.status};
      } catch (error) {
        return {ok: false, status: 0, error: String(error)};
      }
    })()
    """ % json.dumps(ui_url)
)
if not proxy_result.get("ok"):
    raise RuntimeError(f"Colab proxy URL check failed: {proxy_result}")

print("Temporary Colab UI URL:", ui_url)
display(HTML(f'<p><a href="{ui_url}" target="_blank" rel="noopener noreferrer">Open Expandiffusion UI</a></p>'))

In [ ]:
if "EXPANDIFFUSION_PROCESS" in globals() and EXPANDIFFUSION_PROCESS.poll() is None:
    EXPANDIFFUSION_PROCESS.terminate()
    EXPANDIFFUSION_PROCESS.wait(timeout=20)
    print("Expandiffusion stopped.")
else:
    print("No running Expandiffusion process found.")